# Interpretation Agent — 2×2 Evaluation

Compares extraction accuracy across **2 models × 2 prompts**.

| | llama-3.3-70b (baseline) | DeepSeek-R1 (stronger) |
|---|---|---|
| **Minimal prompt** | ? | ? |
| **Full prompt** | ? | ? |

**Hypothesis**: reasoning models recover accuracy on the minimal prompt that the baseline model loses.

In [13]:
# !pip install langchain-ollama
# import sys
# !{sys.executable} -m pip install langchain-openai
#!pip install langchain-google-genai

## Cell 1 — Schema definitions
Copied from `interpretation_agent.ipynb`. Keep in sync if the main notebook changes.

In [14]:
import os
import re
import json
import time
from typing import Optional, Union
from pydantic import BaseModel, Field, field_validator
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
#import langchain_openai
#from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

class ExtractionResult(BaseModel):
    """Structured clinical facts extracted from a single caregiver message."""
    fever:               Optional[str]   = Field(None, description="'yes' or 'no'")
    temperature_f:       Optional[float] = Field(None, description="Temperature in °F if stated")
    fever_duration_days: Optional[float] = Field(None, description="Days fever has been present")
    vomiting:            Optional[str]   = Field(None, description="'none', 'once', or 'repeated'")
    alert:               Optional[str]   = Field(None, description="'yes' if child responds normally, 'no' if lethargic/hard to wake")
    drinking:            Optional[str]   = Field(None, description="'yes', 'some', or 'no'")
    urination_8h:        Optional[str]   = Field(None, description="'yes' or 'no' — has child urinated in last 8 hours")
    breathing_issues:    Optional[str]   = Field(None, description="'yes' or 'no'")
    rash:                Optional[str]   = Field(None, description="'yes' or 'no'")
    neck_pain:           Optional[str]   = Field(None, description="'yes' or 'no'")
    age_months:          Optional[int]   = Field(None, description="Child age in months")
    current_medication:  Optional[str]   = Field(None, description="Name of medication if mentioned")
    medication_last_dose:Optional[str]   = Field(None, description="When last dose was given")

    # yes/no string fields: coerce bool/int → "yes"/"no"
    @field_validator("fever", "alert", "urination_8h", "breathing_issues", "rash", "neck_pain", mode="before")
    @classmethod
    def coerce_bool_to_yesno(cls, v):
        if isinstance(v, (bool, int)):
            return "yes" if v else "no"
        return v

    @field_validator("drinking", mode="before")
    @classmethod
    def coerce_bool_drinking(cls, v):
        if isinstance(v, (bool, int)):
            return "yes" if v else "no"
        return v

    @field_validator("vomiting", mode="before")
    @classmethod
    def coerce_bool_vomiting(cls, v):
        if isinstance(v, (bool, int)):
            return "once" if v else "none"
        return v

    # Free-text fields: bool/int doesn't map to a meaningful string — treat as null
    @field_validator("current_medication", "medication_last_dose", mode="before")
    @classmethod
    def coerce_bool_freetext(cls, v):
        if isinstance(v, (bool, int)):
            return None
        return v

    # numeric fields: coerce string → correct type
    @field_validator("temperature_f", "fever_duration_days", mode="before")
    @classmethod
    def coerce_float(cls, v):
        if isinstance(v, str):
            try: return float(v)
            except (ValueError, TypeError): return None
        return v

    @field_validator("age_months", mode="before")
    @classmethod
    def coerce_int(cls, v):
        if isinstance(v, str):
            try: return int(float(v))
            except (ValueError, TypeError): return None
        return v

# Clinical fields to compare (excludes agent-logic fields)
CLINICAL_FIELDS = [
    "fever", "temperature_f", "fever_duration_days", "vomiting",
    "alert", "drinking", "urination_8h", "breathing_issues",
    "rash", "neck_pain", "age_months", "current_medication", "medication_last_dose"
]

REQUIRED_FIELDS_ORDER = ["alert", "breathing_issues", "temperature_f", "urination_8h", "drinking", "current_medication"]
QUESTION_MAP = {
    "alert":              "Is your child awake and responding normally when you talk to them?",
    "breathing_issues":   "Is your child having any trouble breathing or breathing fast?",
    "temperature_f":      "What is the temperature right now?",
    "urination_8h":       "Has your child urinated in the last 8 hours?",
    "drinking":           "Is your child drinking any fluids?",
    "current_medication": "Is your child currently taking any medications?",
}

print("Schema definitions loaded.")

Schema definitions loaded.


## Cell 2 — Model configs
Both models initialized here. Swap the stronger model if needed.

In [15]:
from langchain_groq import ChatGroq

GROQ_API_KEY = os.environ["GROQ_API_KEY"]

# Baseline model (stable)
llm_baseline = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=GROQ_API_KEY, temperature=0)

print("Baseline model ready: llama-3.3-70b-versatile")

Baseline model ready: llama-3.3-70b-versatile


## Cell 3 — Prompt variants

**Full prompt**: all explicit rules — tests instruction-following.  
**Minimal prompt**: basic task only — tests whether reasoning fills the gap.

In [16]:
PROMPT_FULL = """You are a clinical information extractor for a pediatric triage system.

YOUR ONLY JOB is to extract structured clinical facts from the caregiver's message.

STRICT RULES:
1. Extract ONLY what the caregiver explicitly states or clearly implies.
2. Set a field to null if the message provides NO new information about it.
3. Do NOT infer, guess, or fill in fields that were not mentioned.
4. Do NOT generate clinical advice, diagnosis, treatment, or disposition — ever.
5. Age conversion: if given in years, convert to months (e.g. "6 years" → 72).
6. Temperature conversion: if given in Celsius, convert to Fahrenheit.
7. Vomiting: 'none' = no vomiting, 'once' = one episode, 'repeated' = more than once.
"""

PROMPT_MINIMAL = """You are a clinical information extractor for a pediatric triage system.

Extract structured clinical facts from the caregiver's message.
Set a field to null if the message does not provide information about it.
Do NOT generate medical advice or diagnoses.
"""

PROMPTS = {
    "full":    PROMPT_FULL,
    "minimal": PROMPT_MINIMAL,
}

print("Prompts defined:", list(PROMPTS.keys()))

Prompts defined: ['full', 'minimal']


## Cell 4 — Load dataset + evaluation helpers

In [17]:
with open("golden_dataset.json") as f:
    dataset = json.load(f)

targeted_cases = dataset["targeted_cases"]
scenarios      = dataset["scenarios"]
print(f"Loaded {len(targeted_cases)} targeted cases, {len(scenarios)} scenarios.")


def run_extraction(caregiver_message: str, history: list, llm, system_prompt: str) -> ExtractionResult:
    """
    Call the LLM directly (no tool calling) to avoid Groq's strict schema validation.
    Parses JSON from the response text and validates with Pydantic.
    Works across all models including DeepSeek-R1 (strips <think> tokens).
    """
    field_list = ", ".join(f'"{f}"' for f in CLINICAL_FIELDS)
    json_instruction = (
        f"\n\nRespond with ONLY a valid JSON object containing these exact fields: {field_list}. "
        "Set any field to null if the message does not mention it."
    )

    prompt_messages = [SystemMessage(content=system_prompt + json_instruction)]
    if history:
        history_text = "\n".join(
            f"{'Caregiver' if role == 'caregiver' else 'Agent'}: {content}"
            for role, content in history
        )
        prompt_messages.append(SystemMessage(content=f"Conversation so far:\n{history_text}"))
    prompt_messages.append(HumanMessage(content=f"Latest message: {caregiver_message}"))

    response = llm.invoke(prompt_messages)
    text = response.content

    # Strip DeepSeek-R1 <think>...</think> reasoning tokens
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Strip markdown code fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text.strip())
    text = re.sub(r"\s*```$", "", text.strip())
    # Extract the first JSON object found
    match = re.search(r"\{.*\}", text, re.DOTALL)
    data = json.loads(match.group()) if match else {}

    return ExtractionResult.model_validate(data)


def simulate_merge(prior_state: dict, extracted: ExtractionResult) -> dict:
    """Merge extracted fields into prior_state, only updating None fields."""
    merged = dict(prior_state)
    for field in CLINICAL_FIELDS:
        new_val = getattr(extracted, field, None)
        if new_val is not None and merged.get(field) is None:
            merged[field] = new_val
    return merged


def compare_fields(expected: dict, actual_source) -> dict:
    """Compare expected vs actual per field. Returns {field: {expected, actual, match}}."""
    results = {}
    for field in CLINICAL_FIELDS:
        if field not in expected:
            continue
        exp = expected[field]
        act = actual_source.get(field) if isinstance(actual_source, dict) else getattr(actual_source, field, None)
        results[field] = {"expected": exp, "actual": act, "match": exp == act}
    return results


def score(comparison: dict) -> tuple:
    correct = sum(1 for r in comparison.values() if r["match"])
    return correct, len(comparison)


def next_required_field(state: dict) -> Optional[str]:
    return next((f for f in REQUIRED_FIELDS_ORDER if state.get(f) is None), None)


print("Helpers defined.")

Loaded 10 targeted cases, 2 scenarios.
Helpers defined.


## Cell 5 — Evaluate function + per-model run cells

`all_results` accumulates across runs — run each model cell independently as needed.  
The summary table (Cell 6) works with however many models have been run.

In [18]:
def evaluate(llm, system_prompt: str) -> dict:
    """Run full evaluation for one (model, prompt) combo."""
    targeted_results = []
    for case in targeted_cases:
        extracted = run_extraction(case["caregiver_message"], [], llm, system_prompt)
        ext_cmp   = compare_fields(case["expected_extracted_fields"], extracted)
        merged    = simulate_merge(case["prior_state"], extracted)
        cum_cmp   = compare_fields(case["expected_cumulative_state"], merged)
        targeted_results.append({
            "id":         case["id"],
            "category":   case["category"],
            "extraction": {"correct": score(ext_cmp)[0], "total": score(ext_cmp)[1], "detail": ext_cmp},
            "cumulative": {"correct": score(cum_cmp)[0], "total": score(cum_cmp)[1], "detail": cum_cmp},
        })

    scenario_results = []
    for scenario in scenarios:
        state, history, turn_results = {f: None for f in CLINICAL_FIELDS}, [], []
        for turn in scenario["turns"]:
            msg       = turn["caregiver_message"]
            extracted = run_extraction(msg, history, llm, system_prompt)
            ext_cmp   = compare_fields(turn["expected_extracted_fields"], extracted)
            state     = simulate_merge(state, extracted)
            cum_cmp   = compare_fields(turn["expected_cumulative_state"], state)
            nf        = next_required_field(state)
            history.append(("caregiver", msg))
            if nf:
                history.append(("agent", QUESTION_MAP[nf]))
            turn_results.append({
                "turn":       turn["turn"],
                "extraction": {"correct": score(ext_cmp)[0], "total": score(ext_cmp)[1]},
                "cumulative": {"correct": score(cum_cmp)[0], "total": score(cum_cmp)[1]},
            })
        scenario_results.append({"id": scenario["id"], "turns": turn_results})

    return {"targeted": targeted_results, "scenarios": scenario_results}


# Accumulated results — run each model cell below independently
all_results = {}
all_timings = {}
print("evaluate() defined. Run a model cell below to populate all_results.")

evaluate() defined. Run a model cell below to populate all_results.


In [19]:
# ── Run: llama-3.3-70b (baseline) ──────────────────────────────────────────
MODEL_NAME = "llama-3.3-70b"
for prompt_name, prompt_text in PROMPTS.items():
    key = (MODEL_NAME, prompt_name)
    print(f"Running: {MODEL_NAME} × {prompt_name} ...", end=" ", flush=True)
    t0 = time.time()
    all_results[key] = evaluate(llm_baseline, prompt_text)
    elapsed = time.time() - t0
    all_timings[key] = elapsed
    c = sum(r["extraction"]["correct"] for r in all_results[key]["targeted"])
    t = sum(r["extraction"]["total"]   for r in all_results[key]["targeted"])
    print(f"{c}/{t}  ({elapsed:.1f}s)")
print("llama done.")

Running: llama-3.3-70b × full ... 119/130  (6.5s)
Running: llama-3.3-70b × minimal ... 120/130  (11.3s)
llama done.


In [20]:
# ── Run: gemini-2.5-flash (reasoning) ───────────────────────────────────────────
from langchain_google_genai import ChatGoogleGenerativeAI

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
if not GOOGLE_API_KEY:
    raise ValueError("Set GOOGLE_API_KEY in your .env file")

llm_strong = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0,
)
MODEL_NAME = "gemini-2.5-flash"

for prompt_name, prompt_text in PROMPTS.items():
    key = (MODEL_NAME, prompt_name)
    print(f"Running: {MODEL_NAME} × {prompt_name} ...", end=" ", flush=True)
    t0 = time.time()
    all_results[key] = evaluate(llm_strong, prompt_text)
    elapsed = time.time() - t0
    all_timings[key] = elapsed
    c = sum(r["extraction"]["correct"] for r in all_results[key]["targeted"])
    t = sum(r["extraction"]["total"]   for r in all_results[key]["targeted"])
    print(f"{c}/{t}  ({elapsed:.1f}s)")
print("gemini done.")


Running: gemini-2.5-flash × full ... 119/130  (49.8s)
Running: gemini-2.5-flash × minimal ... 120/130  (40.2s)
gemini done.


## Cell 5.2 — Stripped `json_instruction` experiment

Re-runs the 2×2 matrix with the null-hint removed from `json_instruction`.
This tests whether the original flat results were caused by the hardcoded
`"Set any field to null if the message does not mention it."` that was always
appended regardless of which prompt was used.

| | llama-3.3-70b | gemini-2.5-flash |
|---|---|---|
| **Minimal prompt** | ? | ? |
| **Full prompt** | ? | ? |

In [21]:
def run_extraction_stripped(caregiver_message: str, history: list, llm, system_prompt: str) -> ExtractionResult:
    """
    Same as run_extraction but WITHOUT the null-hint in json_instruction.
    This lets us test whether models differ when not explicitly told to null-fill.
    """
    field_list = ", ".join(f'"{f}"' for f in CLINICAL_FIELDS)
    # Key difference: no 'Set any field to null...' sentence here
    json_instruction = (
        f"\n\nRespond with ONLY a valid JSON object with these exact fields: {field_list}."
    )

    prompt_messages = [SystemMessage(content=system_prompt + json_instruction)]
    if history:
        history_text = "\n".join(
            f"{'Caregiver' if role == 'caregiver' else 'Agent'}: {content}"
            for role, content in history
        )
        prompt_messages.append(SystemMessage(content=f"Conversation so far:\n{history_text}"))
    prompt_messages.append(HumanMessage(content=f"Latest message: {caregiver_message}"))

    response = llm.invoke(prompt_messages)
    text = response.content
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text.strip())
    text = re.sub(r"\s*```$", "", text.strip())
    match = re.search(r"\{.*\}", text, re.DOTALL)
    data = json.loads(match.group()) if match else {}
    return ExtractionResult.model_validate(data)


def evaluate_stripped(llm, system_prompt: str) -> dict:
    """Same as evaluate() but uses run_extraction_stripped."""
    targeted_results = []
    for case in targeted_cases:
        extracted = run_extraction_stripped(case["caregiver_message"], [], llm, system_prompt)
        ext_cmp   = compare_fields(case["expected_extracted_fields"], extracted)
        merged    = simulate_merge(case["prior_state"], extracted)
        cum_cmp   = compare_fields(case["expected_cumulative_state"], merged)
        targeted_results.append({
            "id":         case["id"],
            "category":   case["category"],
            "extraction": {"correct": score(ext_cmp)[0], "total": score(ext_cmp)[1], "detail": ext_cmp},
            "cumulative": {"correct": score(cum_cmp)[0], "total": score(cum_cmp)[1], "detail": cum_cmp},
        })

    scenario_results = []
    for scenario in scenarios:
        state, history, turn_results = {f: None for f in CLINICAL_FIELDS}, [], []
        for turn in scenario["turns"]:
            msg       = turn["caregiver_message"]
            extracted = run_extraction_stripped(msg, history, llm, system_prompt)
            ext_cmp   = compare_fields(turn["expected_extracted_fields"], extracted)
            state     = simulate_merge(state, extracted)
            cum_cmp   = compare_fields(turn["expected_cumulative_state"], state)
            nf        = next_required_field(state)
            history.append(("caregiver", msg))
            if nf:
                history.append(("agent", QUESTION_MAP[nf]))
            turn_results.append({
                "turn":       turn["turn"],
                "extraction": {"correct": score(ext_cmp)[0], "total": score(ext_cmp)[1]},
                "cumulative": {"correct": score(cum_cmp)[0], "total": score(cum_cmp)[1]},
            })
        scenario_results.append({"id": scenario["id"], "turns": turn_results})

    return {"targeted": targeted_results, "scenarios": scenario_results}


# Accumulated results for the stripped experiment
all_results_stripped = {}
all_timings_stripped = {}

for MODEL_NAME, llm in [("llama-3.3-70b", llm_baseline), ("gemini-2.5-flash", llm_strong)]:
    for prompt_name, prompt_text in PROMPTS.items():
        key = (MODEL_NAME, prompt_name)
        print(f"Running stripped: {MODEL_NAME} × {prompt_name} ...", end=" ", flush=True)
        t0 = time.time()
        all_results_stripped[key] = evaluate_stripped(llm, prompt_text)
        elapsed = time.time() - t0
        all_timings_stripped[key] = elapsed
        c = sum(r["extraction"]["correct"] for r in all_results_stripped[key]["targeted"])
        t = sum(r["extraction"]["total"]   for r in all_results_stripped[key]["targeted"])
        print(f"{c}/{t}  ({elapsed:.1f}s)")

print("Cell 5.2 done.")


Running stripped: llama-3.3-70b × full ... 120/130  (6.2s)
Running stripped: llama-3.3-70b × minimal ... 120/130  (10.5s)
Running stripped: gemini-2.5-flash × full ... 120/130  (50.0s)
Running stripped: gemini-2.5-flash × minimal ... 120/130  (44.3s)
Cell 5.2 done.


## Cell 6 — 2×2x2 summary table

In [22]:
# ── Original (with null-hint) ──────────────────────────────────────────────
prompt_names = list(PROMPTS.keys())
model_names  = sorted({m for m, _ in all_results.keys()})
combo_keys   = [(m, p) for m in model_names for p in prompt_names if (m, p) in all_results]

def accuracy_str(key, results_dict) -> str:
    r = results_dict[key]
    c = sum(x["extraction"]["correct"] for x in r["targeted"])
    t = sum(x["extraction"]["total"]   for x in r["targeted"])
    return f"{c}/{t} ({int(100*c/t)}%)"

def timing_str(key, timings_dict) -> str:
    return f"{timings_dict[key]:.1f}s"

def print_2x2(label: str, results_dict: dict, timings_dict: dict):
    avail_keys   = set(results_dict.keys())
    m_names      = sorted({m for m, _ in avail_keys})
    col_w        = 22
    header       = f"{'':20}" + "".join(f"{m:^{col_w}}" for m in m_names)
    divider      = "-" * (20 + col_w * len(m_names))
    print(f"\n{label}")
    print(divider)
    print(header)
    print(divider)
    for p in prompt_names:
        acc_row  = f"prompt: {p:<12}"
        time_row = f"  time{'':<15}"
        for m in m_names:
            key = (m, p)
            acc_row  += f"{accuracy_str(key, results_dict):^{col_w}}"  if key in avail_keys else f"{'(not run)':^{col_w}}"
            time_row += f"{timing_str(key, timings_dict):^{col_w}}"    if key in avail_keys else f"{'':^{col_w}}"
        print(acc_row)
        print(time_row)
        print(divider)

print_2x2("2×2 RESULTS — with null-hint in json_instruction (original)", all_results, all_timings)

if all_results_stripped:
    print_2x2("2×2 RESULTS — WITHOUT null-hint in json_instruction (Cell 5.2)", all_results_stripped, all_timings_stripped)

# ── Per-case breakdown: side-by-side original vs stripped ───────────────────
if all_results_stripped:
    print("\nPER-CASE BREAKDOWN — original vs stripped (extraction correct/total)")
    headers = []
    combo_pairs = []  # (label, key, results_dict)
    for m in model_names:
        for p in prompt_names:
            key = (m, p)
            if key in all_results:
                combo_pairs.append((f"{m[:8]}/{p[:4]}|orig", key, all_results))
            if key in all_results_stripped:
                combo_pairs.append((f"{m[:8]}/{p[:4]}|strip", key, all_results_stripped))

    col = 17
    print(f"{'Case':<10} {'Category':<22}" + "".join(f"{lbl:^{col}}" for lbl, _, _ in combo_pairs))
    print("-" * (32 + col * len(combo_pairs)))
    for i, case in enumerate(targeted_cases):
        row = f"{case['id']:<10} {case['category']:<22}"
        for lbl, key, rd in combo_pairs:
            r = rd[key]["targeted"][i]
            c, t = r["extraction"]["correct"], r["extraction"]["total"]
            row += f"{str(c)+'/'+str(t)+(' ✓' if c==t else ' ✗'):^{col}}"
        print(row)
    print("-" * (32 + col * len(combo_pairs)))
    totals_row = f"{'TOTAL':<10} {'':22}"
    time_row   = f"{'TIME':<10} {'':22}"
    for lbl, key, rd in combo_pairs:
        c = sum(r["extraction"]["correct"] for r in rd[key]["targeted"])
        t = sum(r["extraction"]["total"]   for r in rd[key]["targeted"])
        totals_row += f"{str(c)+'/'+str(t)+' ('+str(int(100*c/t))+'%)':^{col}}"
        td = all_timings if rd is all_results else all_timings_stripped
        time_row   += timing_str(key, td).center(col)
    print(totals_row)
    print(time_row)
else:
    # Fallback: original per-case breakdown (Cell 5.2 not yet run)
    print("\nPER-CASE BREAKDOWN (extraction correct/total)")
    combo_labels = [f"{m[:10]}/{p[:4]}" for m, p in combo_keys]
    col = 16
    print(f"{'Case':<10} {'Category':<26}" + "".join(f"{l:^{col}}" for l in combo_labels))
    print("-" * (36 + col * len(combo_keys)))
    for i, case in enumerate(targeted_cases):
        row = f"{case['id']:<10} {case['category']:<26}"
        for key in combo_keys:
            r = all_results[key]["targeted"][i]
            c, t = r["extraction"]["correct"], r["extraction"]["total"]
            row += f"{str(c)+'/'+str(t)+(' ✓' if c==t else ' ✗'):^{col}}"
        print(row)
    print("-" * (36 + col * len(combo_keys)))
    totals_row = f"{'TOTAL':<10} {'':<26}"
    time_row   = f"{'TIME':<10} {'':<26}"
    for key in combo_keys:
        c = sum(r["extraction"]["correct"] for r in all_results[key]["targeted"])
        t = sum(r["extraction"]["total"]   for r in all_results[key]["targeted"])
        totals_row += f"{str(c)+'/'+str(t)+' ('+str(int(100*c/t))+'%)':^{col}}"
        time_row   += timing_str(key, all_timings).center(col)
    print(totals_row)
    print(time_row)



2×2 RESULTS — with null-hint in json_instruction (original)
----------------------------------------------------------------
                       gemini-2.5-flash       llama-3.3-70b     
----------------------------------------------------------------
prompt: full            119/130 (91%)         119/130 (91%)     
  time                       49.8s                  6.5s         
----------------------------------------------------------------
prompt: minimal         120/130 (92%)         120/130 (92%)     
  time                       40.2s                 11.3s         
----------------------------------------------------------------

2×2 RESULTS — WITHOUT null-hint in json_instruction (Cell 5.2)
----------------------------------------------------------------
                       gemini-2.5-flash       llama-3.3-70b     
----------------------------------------------------------------
prompt: full            120/130 (92%)         120/130 (92%)     
  time                      

## Cell 7 — Scenario results

In [23]:
print("SCENARIO RESULTS — Cumulative state accuracy per turn")
print(f"{'':30}" + "".join(f"{f'{m[:8]}/{p[:4]}':^14}" for m in model_names for p in prompt_names))
print("-" * (30 + 14 * len(model_names) * len(prompt_names)))

for s_idx, scenario in enumerate(scenarios):
    for t_idx, turn in enumerate(scenario["turns"]):
        label = f"{scenario['id']} turn {turn['turn']}"
        row = f"{label:<30}"
        for m in model_names:
            for p in prompt_names:
                tr = all_results[(m, p)]["scenarios"][s_idx]["turns"][t_idx]
                c, t = tr["cumulative"]["correct"], tr["cumulative"]["total"]
                cell = f"{c}/{t}" + (" ✓" if c == t else " ✗")
                row += f"{cell:^14}"
        print(row)

SCENARIO RESULTS — Cumulative state accuracy per turn
                              gemini-2/full gemini-2/mini llama-3./full llama-3./mini 
--------------------------------------------------------------------------------------
scenario_1 turn 1                12/13 ✗       12/13 ✗       12/13 ✗       12/13 ✗    
scenario_1 turn 2                10/13 ✗       10/13 ✗       11/13 ✗       10/13 ✗    
scenario_1 turn 3                10/13 ✗       10/13 ✗       11/13 ✗       10/13 ✗    
scenario_2 turn 1                12/13 ✗       12/13 ✗       12/13 ✗       12/13 ✗    
scenario_2 turn 2                11/13 ✗       13/13 ✓       13/13 ✓       13/13 ✓    
scenario_2 turn 3                10/13 ✗       13/13 ✓       13/13 ✓       13/13 ✓    


## Cell 8 — Failure detail for any cell

In [24]:
# Inspect a specific (model, prompt) × case combination
INSPECT_MODEL  = "llama-3.3-70b"   # change as needed
INSPECT_PROMPT = "minimal"          # "full" or "minimal"
INSPECT_CASE   = "tc_01"            # tc_01 through tc_10

key = (INSPECT_MODEL, INSPECT_PROMPT)
case_result = next((r for r in all_results[key]["targeted"] if r["id"] == INSPECT_CASE), None)

if case_result is None:
    print(f"Not found: {key} / {INSPECT_CASE}")
else:
    print(f"Model: {INSPECT_MODEL}  Prompt: {INSPECT_PROMPT}  Case: {INSPECT_CASE}")
    print(f"Category: {case_result['category']}\n")
    print(f"{'Field':<25} {'Expected':<15} {'Actual':<15} Result")
    print("-" * 65)
    for field, r in case_result["extraction"]["detail"].items():
        mark = "OK  " if r["match"] else "FAIL"
        print(f"{field:<25} {str(r['expected']):<15} {str(r['actual']):<15} {mark}")

Model: llama-3.3-70b  Prompt: minimal  Case: tc_01
Category: implied-field

Field                     Expected        Actual          Result
-----------------------------------------------------------------
fever                     None            None            OK  
temperature_f             None            None            OK  
fever_duration_days       None            None            OK  
vomiting                  None            None            OK  
alert                     no              no              OK  
drinking                  None            None            OK  
urination_8h              None            None            OK  
breathing_issues          None            None            OK  
rash                      None            None            OK  
neck_pain                 None            None            OK  
age_months                None            None            OK  
current_medication        None            None            OK  
medication_last_dose      None       